# 🛳 Titanic Survival Prediction Project

## 1) Imports & Load Data

In [16]:
import re
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Load CSV (adjust path if needed)
df = pd.read_csv("Titanic-Dataset.csv")
df.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 2) Feature Engineering

In [4]:
def extract_title(name: str) -> str:
    m = re.search(r",\s*([^\.]+)\.", name)
    return m.group(1).strip() if m else "Unknown"

# Title from name
df["Title"] = df["Name"].apply(extract_title)
title_map = {
    "Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs",
    "Lady": "Royalty", "Countess": "Royalty", "Sir": "Royalty",
    "Jonkheer": "Royalty", "Don": "Royalty", "Dona": "Royalty",
    "Dr": "Officer", "Rev": "Officer", "Col": "Officer", "Major": "Officer",
    "Capt": "Officer"
}
df["Title"] = df["Title"].replace(title_map)

# Family features
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

# Cabin flag (since Cabin is mostly missing)
df["HasCabin"] = df["Cabin"].notna().astype(int)


### 3) Select Features & Split

In [5]:
target = "Survived"
drop_cols = ["PassengerId", "Name", "Ticket", "Cabin"]
X = df.drop(columns=[target] + drop_cols)
y = df[target]

numeric_features = ["Age", "SibSp", "Parch", "Fare", "FamilySize"]
categorical_features = ["Pclass", "Sex", "Embarked", "Title", "IsAlone", "HasCabin"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


### 4) Preprocessing Pipeline

In [6]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)


## 5) Build Models

In [7]:
log_reg_clf = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

rf_clf = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1))
])


## 6) Train

In [8]:
log_reg_clf.fit(X_train, y_train)
rf_clf.fit(X_train, y_train)


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Age', 'SibSp', 'Parch',
                                                   'Fare', 'FamilySize']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Pclass', 'Sex', 'Embarked',
                                                   'Title', 'IsAlone',
                                                   'HasCabin'])])),
                ('model',
                 RandomForestClassifier(n_estimators=300, n_jobs=-1,
                                        random_state=42))])

## 7) Evaluate

In [9]:
def evaluate_model(model, X_test, y_test, name="Model"):
    y_pred = model.predict(X_test)
    try:
        y_proba = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_proba)
    except Exception:
        auc = float("nan")
    print(f"=== {name} ===")
    print("Accuracy: ", round(accuracy_score(y_test, y_pred), 4))
    print("Precision:", round(precision_score(y_test, y_pred), 4))
    print("Recall:   ", round(recall_score(y_test, y_pred), 4))
    print("F1-score: ", round(f1_score(y_test, y_pred), 4))
    print("ROC-AUC:  ", round(auc, 4))
    print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

evaluate_model(log_reg_clf, X_test, y_test, "Logistic Regression")
evaluate_model(rf_clf,    X_test, y_test, "Random Forest")


=== Logistic Regression ===
Accuracy:  0.8324
Precision: 0.8
Recall:    0.7536
F1-score:  0.7761
ROC-AUC:   0.8722

Confusion Matrix:
 [[97 13]
 [17 52]]

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.88      0.87       110
           1       0.80      0.75      0.78        69

    accuracy                           0.83       179
   macro avg       0.83      0.82      0.82       179
weighted avg       0.83      0.83      0.83       179

=== Random Forest ===
Accuracy:  0.8156
Precision: 0.7647
Recall:    0.7536
F1-score:  0.7591
ROC-AUC:   0.8325

Confusion Matrix:
 [[94 16]
 [17 52]]

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.85      0.85       110
           1       0.76      0.75      0.76        69

    accuracy                           0.82       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.82      0.82      0.82   

## 8) Hyperparameter Tuning (small, safe grid)

In [11]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 6, 10],
    "model__min_samples_split": [2, 5],
}
grid = GridSearchCV(
    Pipeline([("preprocess", preprocessor),
              ("model", RandomForestClassifier(random_state=42, n_jobs=-1))]),
    param_grid=param_grid, cv=5, scoring="accuracy", n_jobs=-1
)
grid.fit(X_train, y_train)
print("Best Params:", grid.best_params_)
best_model = grid.best_estimator_

# If you skip tuning, just use rf_clf as the best model:
best_model = rf_clf


Best Params: {'model__max_depth': 6, 'model__min_samples_split': 5, 'model__n_estimators': 200}


## 9) Feature Importances (Random Forest)

In [12]:
def rf_feature_importances(rf_pipeline, numeric_features, categorical_features):
    pre = rf_pipeline.named_steps["preprocess"]
    ohe = pre.named_transformers_["cat"].named_steps["onehot"]
    cat_names = []
    for feat, cats in zip(categorical_features, ohe.categories_):
        cat_names.extend([f"{feat}__{c}" for c in cats])
    feature_names = numeric_features + cat_names
    importances = rf_pipeline.named_steps["model"].feature_importances_
    fi = pd.DataFrame({"feature": feature_names, "importance": importances})
    return fi.sort_values("importance", ascending=False)

fi = rf_feature_importances(best_model, numeric_features, categorical_features)
print(fi.head(15))


        feature  importance
3          Fare    0.207674
0           Age    0.200891
8   Sex__female    0.095332
9     Sex__male    0.085883
15    Title__Mr    0.079386
4    FamilySize    0.042644
7     Pclass__3    0.038693
21  HasCabin__0    0.028834
14  Title__Miss    0.027616
22  HasCabin__1    0.026681
1         SibSp    0.025764
16   Title__Mrs    0.022358
2         Parch    0.020972
5     Pclass__1    0.018320
6     Pclass__2    0.016376


## 10) Save the Trained Model

In [17]:
import joblib
joblib.dump(best_model, "titanic_model.joblib")

['titanic_model.joblib']

## 11) Predict on New Data

In [18]:
new_passenger = pd.DataFrame([{
    "Pclass": 3, "Sex": "male", "Age": 22, "SibSp": 1, "Parch": 0,
    "Fare": 7.25, "Embarked": "S", "Title": "Mr", "FamilySize": 2,
    "IsAlone": 0, "HasCabin": 0
}])

pred = best_model.predict(new_passenger)[0]
prob = best_model.predict_proba(new_passenger)[0, 1]
print(f"Predicted Survived = {pred} (probability = {prob:.3f})")


Predicted Survived = 0 (probability = 0.183)
